In [10]:
# ============================================================
# TASK 2: EXPLORATORY DATA ANALYSIS (EDA)
# Adding imports here too — just in case the runtime was reset
# or this cell gets run on its own. Better safe than sorry!
# ============================================================

import pandas as pd          # might already be loaded, but re-importing is harmless
import os

print("=" * 55)
print("       EXPLORATORY DATA ANALYSIS — LET'S DIG IN!")
print("=" * 55)

# Double-checking that our CSV file actually exists before trying to load it
if not os.path.exists("books_raw_data.csv"):
    raise FileNotFoundError(
        " 'books_raw_data.csv' not found! Please run the scraping cells (Cells 1–3) first."
    )

# Load back from CSV (simulating a real pipeline where scraping & EDA are separate steps)
books_df = pd.read_csv("books_raw_data.csv")

       EXPLORATORY DATA ANALYSIS — LET'S DIG IN!


In [11]:
# --- Step 2: Cleaning up the data ---
print("\n🧹 STEP 2: Time to clean things up a bit...\n")

# Let's check for any sneaky missing values hiding in our data
missing_values = books_df.isnull().sum()
print("Missing values per column:")
print(missing_values)

if missing_values.sum() == 0:
    print("\n no missing values found! The scraper did a clean job.")
else:
    print(f"\n  Found {missing_values.sum()} missing values. Dropping those rows...")
    books_df.dropna(inplace=True)

# Check for duplicates
num_duplicates = books_df.duplicated(subset=["title"]).sum()
print(f"\n Duplicate book titles found: {num_duplicates}")
if num_duplicates > 0:
    print("   Dropping duplicates and keeping the first occurrence...")
    books_df.drop_duplicates(subset=["title"], keep="first", inplace=True)
    print(f"   Dataset size after removing duplicates: {books_df.shape[0]} books")

# ---------------------------------------------------------------
# FIX: The £ symbol gets mangled to 'Â£' when saved/read from CSV
# due to a UTF-8 encoding mismatch. Instead of matching a specific
# character, we strip out ALL non-numeric characters except the dot.
# This is more robust and handles any encoding weirdness gracefully.
# ---------------------------------------------------------------
print("\n Cleaning price column — stripping currency symbols and converting to float...")

# Let's see what the raw values actually look like before cleaning
print("   Sample raw price values:", books_df["price_raw"].head(3).tolist())

# Use regex to keep only digits and decimal points — works regardless of encoding
books_df["price"] = books_df["price_raw"].str.replace(r"[^\d.]", "", regex=True).astype(float)

# Sanity check — make sure the conversion worked
print(f"   Price column converted! Sample values: {books_df['price'].head(3).tolist()}")

# We don't need the raw price string anymore
books_df.drop(columns=["price_raw"], inplace=True)

# Standardize the availability column — just make it True/False
books_df["in_stock"] = books_df["availability"].str.contains("In stock", case=False)
books_df.drop(columns=["availability"], inplace=True)

print("\n Cleaning done! Here's our tidy dataset:")
print(books_df.head())
print(f"\nFinal dataset shape: {books_df.shape}")


🧹 STEP 2: Time to clean things up a bit...

Missing values per column:
title           0
price_raw       0
rating          0
availability    0
category        0
dtype: int64

 no missing values found! The scraper did a clean job.

 Duplicate book titles found: 0

 Cleaning price column — stripping currency symbols and converting to float...
   Sample raw price values: ['Â£47.82', 'Â£19.63', 'Â£56.50']
   Price column converted! Sample values: [47.82, 19.63, 56.5]

 Cleaning done! Here's our tidy dataset:
                                             title  rating category  price  \
0                                    Sharp Objects       4  Mystery  47.82   
1                             In a Dark, Dark Wood       1  Mystery  19.63   
2                              The Past Never Ends       4  Mystery  56.50   
3                                 A Murder in Time       1  Mystery  16.64   
4  The Murder of Roger Ackroyd (Hercule Poirot #4)       4  Mystery  44.10   

   in_stock  
0     

In [12]:
# --- Step 3: Let's ask some real questions about our data ---
print("\n" + "=" * 55)
print("  ASKING MEANINGFUL QUESTIONS ABOUT OUR DATASET")
print("=" * 55)

# Question 1: What's the price range we're dealing with?
min_price = books_df["price"].min()
max_price = books_df["price"].max()
avg_price = books_df["price"].mean()
print(f"\n Q1: What's the price range of books in our dataset?")
print(f"   → Cheapest book:  £{min_price:.2f}")
print(f"   → Most expensive: £{max_price:.2f}")
print(f"   → Average price:  £{avg_price:.2f}")

# Question 2: How are books distributed across categories?
print(f"\n Q2: How many books did we collect per category?")
category_counts = books_df["category"].value_counts()
print(category_counts.to_string())

# Question 3: Which category has the highest average rating?
print(f"\n Q3: Which category tends to get the best reader ratings on average?")
avg_rating_by_category = books_df.groupby("category")["rating"].mean().sort_values(ascending=False)
print(avg_rating_by_category.round(2).to_string())
top_rated_category = avg_rating_by_category.idxmax()
print(f"\n   → The best-rated category is: '{top_rated_category}' ⭐")

# Question 4: What does the rating distribution look like?
print(f"\n Q4: How are ratings spread out across all books?")
rating_distribution = books_df["rating"].value_counts().sort_index()
print(rating_distribution.to_string())

# Question 5: Any expensive books with surprisingly low ratings?
print(f"\n Q5: Are there any pricey books (over £40) with a rating of 2 or below?")
poor_value_books = books_df[(books_df["price"] > 40) & (books_df["rating"] <= 2)]
if poor_value_books.empty:
    print("   → None found! Expensive books seem to be well-rated in our dataset.")
else:
    print(poor_value_books[["title", "price", "rating", "category"]])

# Save the cleaned data for use in visualization
books_df.to_csv("books_cleaned_data.csv", index=False)
print("\n Cleaned dataset saved to 'books_cleaned_data.csv'")
print("\n EDA complete! We have a solid understanding of our data now.")


  ASKING MEANINGFUL QUESTIONS ABOUT OUR DATASET

 Q1: What's the price range of books in our dataset?
   → Cheapest book:  £10.65
   → Most expensive: £59.99
   → Average price:  £34.40

 Q2: How many books did we collect per category?
category
Romance            35
Mystery            32
History            18
Science Fiction    16
Travel             11

 Q3: Which category tends to get the best reader ratings on average?
category
History            2.94
Mystery            2.94
Travel             2.73
Romance            2.63
Science Fiction    2.25

   → The best-rated category is: 'History' ⭐

 Q4: How are ratings spread out across all books?
rating
1    30
2    21
3    26
4    20
5    15

 Q5: Are there any pricey books (over £40) with a rating of 2 or below?
                                                 title  price  rating  \
5                       The Last Mile (Amos Decker #2)  54.21       2   
31                 1st to Die (Women's Murder Club #1)  53.98       1   
37       